# TFM Tennis CV — Pipeline completo (Google Colab)

Ejecuta las celdas en orden. El dataset y los outputs se leen/escriben en tu Google Drive.

**Rutas esperadas en Drive:**
```
Mi unidad/TFM/
├── Dataset/          ← gameN/ClipM/*.jpg + Label.csv
├── outputs/
│   └── models/
│       └── tracknet_best.pth
└── ProyectoTFM/      ← código del repo (se clona aquí si no existe)
```

## 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuración de rutas

Ajusta `DRIVE_TFM` si tu carpeta TFM está en una ruta distinta dentro de Drive.

In [ ]:
import os

DRIVE_TFM     = '/content/drive/MyDrive/TFM'
REPO_DIR      = f'{DRIVE_TFM}/ProyectoTFM'
DATASET_ROOT  = f'{DRIVE_TFM}/Dataset'
OUTPUTS_ROOT  = f'{DRIVE_TFM}/outputs'

# Variables de entorno que config.py lee para sobreescribir rutas locales.
os.environ['TFM_PROJECT_ROOT'] = REPO_DIR
os.environ['TFM_DATASET_ROOT'] = DATASET_ROOT
os.environ['TFM_OUTPUTS_ROOT'] = OUTPUTS_ROOT

print('Drive TFM :', DRIVE_TFM)
print('Repo      :', REPO_DIR)
print('Dataset   :', DATASET_ROOT)
print('Outputs   :', OUTPUTS_ROOT)

## 3. Clonar / actualizar el repositorio

In [ ]:
import os
from pathlib import Path

if Path(REPO_DIR).exists():
    print('Repo ya existe, actualizando...')
    !cd "{REPO_DIR}" && git pull
else:
    # Sustituye la URL por la de tu repo si es privado (usa token o SSH).
    REPO_URL = 'https://github.com/TU_USUARIO/ProyectoTFM.git'
    !git clone "{REPO_URL}" "{REPO_DIR}"

os.chdir(REPO_DIR)
print('Directorio de trabajo:', os.getcwd())

## 4. Instalar dependencias

In [ ]:
# Colab ya trae torch/torchvision con CUDA. Solo instalamos las extras.
!pip install -q ultralytics lap openpyxl

# Verificar GPU
import torch
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 5. Verificar estructura antes de ejecutar

In [ ]:
from pathlib import Path

checks = {
    'Dataset':           Path(DATASET_ROOT),
    'tracknet_best.pth': Path(OUTPUTS_ROOT) / 'models' / 'tracknet_best.pth',
    'config.py':         Path(REPO_DIR) / 'config.py',
    'main.py':           Path(REPO_DIR) / 'main.py',
}

all_ok = True
for nombre, ruta in checks.items():
    ok = ruta.exists()
    print(f"  {'OK' if ok else 'FALTA'}  {nombre}: {ruta}")
    all_ok = all_ok and ok

if all_ok:
    print('\nTodo listo. Puedes ejecutar la celda siguiente.')
else:
    print('\nFaltan rutas. Revisa la celda de configuracion.')

## 6. Ejecutar el pipeline

Cambia `--game-path` al game que quieras procesar. Repite la celda para más games.

In [ ]:
GAME = 'game1'  # <-- cambia aquí

!python main.py \
    --game-path "{DATASET_ROOT}/{GAME}" \
    --output-dir "{OUTPUTS_ROOT}/{GAME}" \
    --log-level INFO

## 7. (Opcional) Procesar todos los games en bucle

In [ ]:
from pathlib import Path

games = sorted(Path(DATASET_ROOT).glob('game*'))
print(f'Games encontrados: {[g.name for g in games]}')

for game_path in games:
    print(f'\n=== Procesando {game_path.name} ===')
    out = Path(OUTPUTS_ROOT) / game_path.name
    !python main.py \
        --game-path "{game_path}" \
        --output-dir "{out}" \
        --log-level INFO

## 8. Entrenamiento TrackNet (mejora del modelo)

Ejecuta esta celda solo si quieres reentrenar o afinar el modelo.  
El checkpoint resultante se guarda en `outputs/models/tracknet_best.pth` en Drive.

In [ ]:
# Abre el notebook de entrenamiento directamente en Colab.
# Ajusta las rutas en la celda '## 1. Configuración' del notebook.
print('Notebook de entrenamiento:', f'{REPO_DIR}/old_notebooks/02-0_model_ball_train.ipynb')
print()
print('Parámetros clave a revisar en ese notebook:')
print(f'  DATASET_ROOT  = "{DATASET_ROOT}"')
print(f'  MODEL_OUT_DIR = "{OUTPUTS_ROOT}/models"')